# 01. Construcción de la base integrada - Fashion Transparency Index 2023

### Objetivo del notebook

Construir una base de datos a partir de las descargas parciales del **Fashion Transparency Index 2023** realizadas desde WikiRate.

### Descripción general

Debido a que la plataforma limita la exportación a un máximo de 5,000 observaciones por descarga, inicialmente la información se recuperó mediante filtros por categoría (Environment, Social y Governance), organizando los archivos en las carpetas correspondientes dentro de `raw/`. Posteriormente, se identificó que esta estrategia no recuperaba la totalidad de las métricas del índice, por lo que las métricas restantes fueron descargadas en bloques adicionales y almacenadas en la carpeta `raw/Other`.

Finalmente, todos los archivos fueron verificados, integrados y depurados para obtener una única base de datos consolidada, comprobando la consistencia de las variables, la estructura de los registros y la ausencia de observaciones duplicadas. Esta base constituirá el punto de partida para las etapas posteriores de limpieza, selección de variables y análisis exploratorio de datos.

In [1]:
# Importamos las librerías necesarias 

import pandas as pd
from pathlib import Path

# Configuración para visualizar mejor las tablas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

### Localización de archivos ordenados por categoría

In [2]:
# Definimos la ruta donde se encuentran los archivos descargados
ruta_datos = Path("/Users/avrilsalazar/Documents/FashionTransparency2023/raw")

# Definimos las carpetas de cada categoría
carpetas = [
    "Environment",
    "Social",
    "Governance"
]

# Creamos una lista vacía para almacenar los archivos
archivos_csv = []

# Buscamos los archivos de cada categoría
for carpeta in carpetas:
    archivos_csv.extend(
        sorted((ruta_datos / carpeta).glob("*.csv"))
    )

# Mostramos el total de archivos encontrados
print(f"Se encontraron {len(archivos_csv)} archivos CSV.\n")

# Mostramos el nombre de cada archivo
for archivo in archivos_csv:
    print(archivo)

Se encontraron 18 archivos CSV.

/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Biodiversity.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Circular_Economy.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Climate_Change.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Energy.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Sustainable_Supply_Chains.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Waste.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Water.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Social/Community_Impact.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Social/Diversity_Equity_Inclusion.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Social/Human_Rights.csv
/Users/avrilsalazar/Documents/FashionTransparency2023/raw/Social/Just_Transition.csv
/Users/avr

### Verificación de la estructura de los archivos

Antes de integrar la información, se verificó que todos los archivos descargados compartieran la misma estructura de variables. Esta comprobación permitió confirmar que la integración de las descargas temáticas podía realizarse de forma consistente mediante la concatenación de los registros.

In [3]:
# Seleccionamos el primer archivo de la lista para revisar su estructura

archivo_ejemplo = archivos_csv[0]

print(f"Archivo revisado: {archivo_ejemplo}\n")

# Mostramos sus primeras líneas para localizar el encabezado real

with open(archivo_ejemplo, "r", encoding="utf-8-sig") as archivo_texto:
    for numero, linea in enumerate(archivo_texto):
        print(f"{numero}: {linea.strip()}")

        if numero == 9:
            break

Archivo revisado: /Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Biodiversity.csv

0: # https://wikirate.org/Fashion_Transparency_Index_2023_full_dataset+Answer?filter%5Bcompany_keyword%5D=&filter%5Bcompany_identifier%5D%5Btype%5D=&filter%5Bcompany_identifier%5D%5Bvalue%5D=&filter%5Bmetric_keyword%5D=&filter%5Btopic%5D%5B%5D=%7E21475062&filter%5Bvalue%5D=&filter%5Bstatus%5D=exists&export_type=answer&format=csv&view=detailed&limit=5000&utf8=%E2%9C%93
1: "# Wikirate.org, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0). See https://wikirate.org/Attribution_Guide."
2: # 2026-07-22 21:37:57 UTC
3: #
4: Answer Page,Metric,Company,Year,Value,Source Page,Answer ID,Original Source,Source Count,Comments,ISIN
5: https://wikirate.org/Fashion_Revolution+Fashion_Transparency_Index_2023+Puma+2023,Fashion Revolution+Fashion Transparency Index 2023,Puma,2023,6.6399545215046025,,,,,,DE0006969603;US7458781082;US7458782072
6: https://wikirate.org/Fashion_Revol

In [4]:
# Revisamos la estructura de cada archivo antes de integrarlos

estructura_archivos = []

# Recorremos todos los archivos descargados
for archivo in archivos_csv:

    # Omitimos las primeras 4 líneas informativas
    df = pd.read_csv(archivo, skiprows=4)

    # Guardamos un resumen de la estructura del archivo
    estructura_archivos.append({
        "archivo": archivo.name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "variables": tuple(df.columns)
    })

# Convertimos el resumen en un DataFrame
estructura = pd.DataFrame(estructura_archivos)

# Mostramos el resumen de cada archivo
estructura[["archivo", "filas", "columnas"]]

,archivo,filas,columnas
0,Biodiversity.csv,750,11
1,Circular_Economy.csv,2000,11
2,Climate_Change.csv,2250,11
3,Energy.csv,1000,11
4,Sustainable_Supply_Chains.csv,2250,11
5,Waste.csv,1250,11
6,Water.csv,1250,11
7,Community_Impact.csv,750,11
8,Diversity_Equity_Inclusion.csv,750,11
9,Human_Rights.csv,1250,11


In [5]:
# Verificamos cuántas estructuras de columnas distintas existen

estructura["variables"].nunique()

1

In [6]:
# Integramos todos los archivos ordenados por categoría en una sola base

lista_dataframes = []

# Recorremos cada archivo descargado
for archivo in archivos_csv:

    # Leemos el archivo omitiendo las líneas informativas
    df = pd.read_csv(archivo, skiprows=4)

    # Registramos la categoría temática del archivo
    df["Categoria"] = archivo.parent.name

    # Agregamos el DataFrame a la lista
    lista_dataframes.append(df)

# Unimos todos los DataFrames
base_integrada = pd.concat(lista_dataframes, ignore_index=True)

# Mostramos el tamaño de la base integrada
print(f"Número total de registros: {len(base_integrada):,}")
print(f"Número de variables: {base_integrada.shape[1]}")

Número total de registros: 27,000
Número de variables: 12


In [7]:
# Revisamos cuántas empresas, indicadores y años contiene la base integrada

print(f"Empresas únicas: {base_integrada['Company'].nunique()}")
print(f"Indicadores únicos: {base_integrada['Metric'].nunique()}")
print(f"Años disponibles: {sorted(base_integrada['Year'].dropna().unique())}")

Empresas únicas: 250
Indicadores únicos: 46
Años disponibles: [2023]


In [8]:
# Obtenemos la lista de métricas presentes en las descargas por categoría

metricas_actuales = sorted(base_integrada["Metric"].dropna().unique())

print(f"Métricas actuales: {len(metricas_actuales)}")

for metrica in metricas_actuales:
    print(metrica)

Métricas actuales: 46
Fashion Revolution+2. Governance Score
Fashion Revolution+4. Know, Show & Fix Score
Fashion Revolution+5. Spotlight Issues Score (2023)
Fashion Revolution+Approach to Defining Sustainable Materials
Fashion Revolution+Decarbonisation Commitment
Fashion Revolution+Decarbonisation Progress
Fashion Revolution+Describes Environmental Due Diligence Process
Fashion Revolution+Discloses Absolute Energy Reduction
Fashion Revolution+Discloses Annual Investment in Decarbonisation
Fashion Revolution+Discloses Breakdown of Reuse/Recyling of Pre-consumer Waste
Fashion Revolution+Discloses Coal Use
Fashion Revolution+Discloses Content of Scope 1, 2 and 3 Emissions
Fashion Revolution+Discloses Efforts to Invest in Supply Chain Workers
Fashion Revolution+Discloses Free on Board (FOB) Price Changes (COVID-19 Response)
Fashion Revolution+Discloses Number of Collective Bargaining Agreements Providing Wages Above Legal Minimum
Fashion Revolution+Discloses Number of Workers Affected by

### Comparación con las métricas del índice

La base integrada contenía información correspondiente a 46 de las 130 métricas del Fashion Transparency Index 2023. Por ello, fue necesario identificar las métricas faltantes para completar la base de datos.

In [9]:
# Registramos la lista completa de métricas del Fashion Transparency Index 2023

texto_metricas = """
Metric
Fashion Revolution+Supply Chain Policies
Fashion Revolution+Supply Chain Policies Align with International Standards
Fashion Revolution+Supply Chain Policies Are Contractual
Fashion Revolution+Supply Chain Policies in Local Language
Fashion Revolution+1.3 Management Procedures
Fashion Revolution+Plan for Improving Human Rights Impacts
Fashion Revolution+Plan for Improving Environmental Impacts
Fashion Revolution+Reports on Efforts to Improve Human Rights Impacts
Fashion Revolution+Reports on Efforts to Improve Environmental Impacts
Fashion Revolution+1.5 Verified Sustainability Report
Fashion Revolution+2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues
Fashion Revolution+Accountable Board Member Identified
Fashion Revolution+Implementation of Board Level Accountability Described
Fashion Revolution+Worker Representation on Board
Fashion Revolution+Responsible Tax Strategy
Fashion Revolution+Employee Incentives to Improve Impacts
Fashion Revolution+Executive Incentives to Improve Impacts
Fashion Revolution+Executive Pay Linked to Environmental and Social Targets
Fashion Revolution+Supplier Incentives to Improve Impacts
Fashion Revolution+3.1 Tier One Factory Disclosure
Fashion Revolution+3.2 Processing Facilities Disclosure
Fashion Revolution+3.3 Raw Materials Suppliers Disclosure
Fashion Revolution+Describes Human Rights Due Diligence Process
Fashion Revolution+Stakeholder Engagement in Human Rights Due Diligence
Fashion Revolution+Approach to Involving Women in Human Rights Due Diligence
Fashion Revolution+Human Rights Risks Impacts and Violations Identified
Fashion Revolution+Prevention Mitigation and Remediation of Human Rights Risks
Fashion Revolution+Human Rights Risk Prevention and Remediation Outcomes Published
Fashion Revolution+Describes Environmental Due Diligence Process
Fashion Revolution+Stakeholder Engagement in Environmental Due Diligence
Fashion Revolution+Environmental Risks Impacts and Violations Identified
Fashion Revolution+Prevention Mitigation and Remediation of Environmental Risks
Fashion Revolution+Environmental Risk Prevention and Remediation Outcomes Published
Fashion Revolution+Scope, Process and Accreditation for Environmental Audits
Fashion Revolution+New Production Facility Criteria
Fashion Revolution+Number or % of off-site worker interviews
Fashion Revolution+Percentage of Audits including Trade Union Representative
Fashion Revolution+Summary of Assessment Findings
Fashion Revolution+Ratings by Named Facilities
Fashion Revolution+Selected Audit Findings by Named Facilities
Fashion Revolution+Full Audit Reports by Named Facilities
Fashion Revolution+Remediation Process
Fashion Revolution+Affected Stakeholder Engagement in Remediation
Fashion Revolution+Exit Strategy
Fashion Revolution+Grievance Mechanism - Direct Employees
Fashion Revolution+Grievance Mechanism - Supply Chain Workers
Fashion Revolution+Grievance Mechanism Implementation - Supply Chain Workers
Fashion Revolution+Grievance Mechanism Disseminated to Supply Chain Workers
Fashion Revolution+Grievance Mechanism in Supplier Policies
Fashion Revolution+Grievance Reporting - Supply Chain Workers
Fashion Revolution+Discloses Approach to Recruitment Fees
Fashion Revolution+Discloses Number of Workers Affected by Recruitment Fees
Fashion Revolution+Discloses Data on Modern Slavery Prevalence
Fashion Revolution+Discloses Approach to Living Wage
Fashion Revolution+Discloses Strategy to Achieving Living Wage
Fashion Revolution+Discloses Progress toward the payment of a Living Wage to workers in the supply chain
Fashion Revolution+Discloses Living Wage Estimates Used for Benchmarking
Fashion Revolution+Discloses Percentage of Workers Receiving Wage Payments Digitally
Fashion Revolution+Publishes Percentage of Workers Paid Above Minimum Wage
Fashion Revolution+Discloses Percentage of Workers Paid By Piece Rate
Fashion Revolution+Reports on Minimum Wage Paid for Daily / Piece Rate Workers
Fashion Revolution+Discloses Proportion of Workers Paid Minimum Wage
Fashion Revolution+Publishes Percentage or Number of Workers Earning a Living Wage
Fashion Revolution+Protects Labour Costs in Price Negotiations
Fashion Revolution+Discloses Quantity of Orders with Labor Cost Protection
Fashion Revolution+Discloses Free on Board (FOB) Price Changes (COVID-19 Response)
Fashion Revolution+Publishes Standard Supplier Agreement Template
Fashion Revolution+Discloses Policy on Up-Front Supplier Payments
Fashion Revolution+Policy to Pay Supplier Within 60 Days
Fashion Revolution+Discloses Time Taken to Pay Purchase Orders
Fashion Revolution+Discloses Quantity of Orders Changed After Original Agreement
Fashion Revolution+Publishes Supplier Feedback on Purchasing Practices
Fashion Revolution+Discloses number or % of supplier facilities that have independent, democratically elected trade unions
Fashion Revolution+Discloses number or % of Workers covered by Collective Bargaining Agreements
Fashion Revolution+Discloses Number of Collective Bargaining Agreements Providing Wages Above Legal Minimum
Fashion Revolution+Discloses Prevalance of Collective Bargaining Violations
Fashion Revolution+Publishes Gender Pay Gap
Fashion Revolution+Publishes Sex-disaggregated Job Distribution
Fashion Revolution+Publishes Gender-based Labour Violations Data
Fashion Revolution+Discloses Gender Equality Actions in Supplier Facilities
Fashion Revolution+Publishes Ethnicity Pay Gap
Fashion Revolution+Publishes Race-disaggregated Job Distribution
Fashion Revolution+Publishes Racial Equality Actions
Fashion Revolution+Discloses Sourced Fibre Breakdown
Fashion Revolution+Sustainable Materials Strategy
Fashion Revolution+Discloses Progress on Sustainable Materials Strategy
Fashion Revolution+Approach to Defining Sustainable Materials
Fashion Revolution+Targets to Reduce Textiles Derived from Virgin Fossil Fuels
Fashion Revolution+Discloses Progress to Reducing Textiles Derived from Virgin Fossil Fuels
Fashion Revolution+Targets to Reduce Virgin Plastics
Fashion Revolution+Discloses Progress to Reducing Virgin Plastics
Fashion Revolution+Minimizing Impact of Microfibres
Fashion Revolution+Discloses Quantity of Products Produced
Fashion Revolution+Commitment to Degrowth
Fashion Revolution+Discloses Quantity of Pre-Production Waste Generated
Fashion Revolution+Discloses Quantity of Post-production Waste Generated
Fashion Revolution+Discloses Breakdown of Reuse/Recyling of Pre-consumer Waste
Fashion Revolution+Discloses Quantity of Products Destroyed
Fashion Revolution+Offers Take-back Schemes
Fashion Revolution+Discloses Take-back Scheme Outcomes
Fashion Revolution+Offers Clothing Longevity Business Models
Fashion Revolution+Offers Repair Services
Fashion Revolution+Discloses Evidence of Developing Circular Solutions
Fashion Revolution+Discloses % of Circular Products
Fashion Revolution+Discloses Efforts to Invest in Supply Chain Workers
Fashion Revolution+Commitment to Eliminate Hazardous Chemicals
Fashion Revolution+Discloses Progress to Eliminate Hazardous Chemicals
Fashion Revolution+Publishes Supplier Wastewater Test Results
Fashion Revolution+Discloses Water Use
Fashion Revolution+Discloses Water-Related Risk Assessment Process
Fashion Revolution+Decarbonisation Commitment
Fashion Revolution+Science Based Targets
Fashion Revolution+Decarbonisation Progress
Fashion Revolution+Discloses Annual Investment in Decarbonisation
Fashion Revolution+Discloses Content of Scope 1, 2 and 3 Emissions
Fashion Revolution+Environmental Profit and Loss Statement
Fashion Revolution+Zero Deforestation Commitment
Fashion Revolution+Zero Deforestation Progress
Fashion Revolution+Implementation of Regenerative Farming Practices
Fashion Revolution+Discloses Absolute Energy Reduction
Fashion Revolution+Discloses Renewable Energy Use
Fashion Revolution+Discloses Coal Use
Fashion Revolution+1.1 Own Operations Policies
Fashion Revolution+Publishes Responsible Purchasing Code of Conduct
Fashion Revolution+1. Policy & Commitments Score
Fashion Revolution+2. Governance Score
Fashion Revolution+4. Know, Show & Fix Score
Fashion Revolution+3. Traceability Score (2023)
Fashion Revolution+5. Spotlight Issues Score (2023)
Fashion Revolution+Fashion Transparency Index 2023
"""

# Convertimos el texto en una lista limpia

metricas_completas = [
    linea.strip()
    for linea in texto_metricas.splitlines()
    if linea.strip() and linea.strip() != "Metric"
]

# Identificamos las métricas que todavía no están en nuestra base

conjunto_actuales = set(metricas_actuales)

metricas_faltantes = [
    metrica
    for metrica in metricas_completas
    if metrica not in conjunto_actuales
]

print(f"Métricas completas: {len(metricas_completas)}")
print(f"Métricas actuales: {len(metricas_actuales)}")
print(f"Métricas faltantes: {len(metricas_faltantes)}")

Métricas completas: 130
Métricas actuales: 46
Métricas faltantes: 84


In [10]:
# Dividimos las métricas faltantes en bloques de máximo 20 (5,000 registros por bloque)

bloques_metricas = [
    metricas_faltantes[i:i + 20]
    for i in range(0, len(metricas_faltantes), 20)
]

# Mostramos el contenido de cada bloque

for numero, bloque in enumerate(bloques_metricas, start=1):
    print(f"\nBLOQUE {numero}: {len(bloque)} métricas")
    
    for metrica in bloque:
        print(metrica)


BLOQUE 1: 20 métricas
Fashion Revolution+Supply Chain Policies
Fashion Revolution+Supply Chain Policies Align with International Standards
Fashion Revolution+Supply Chain Policies Are Contractual
Fashion Revolution+Supply Chain Policies in Local Language
Fashion Revolution+1.3 Management Procedures
Fashion Revolution+Plan for Improving Human Rights Impacts
Fashion Revolution+Plan for Improving Environmental Impacts
Fashion Revolution+Reports on Efforts to Improve Human Rights Impacts
Fashion Revolution+Reports on Efforts to Improve Environmental Impacts
Fashion Revolution+1.5 Verified Sustainability Report
Fashion Revolution+2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues
Fashion Revolution+Accountable Board Member Identified
Fashion Revolution+Implementation of Board Level Accountability Described
Fashion Revolution+Employee Incentives to Improve Impacts
Fashion Revolution+Executive Incentives to Improve Impacts
Fashion Revolution+Supplier Incentives to Imp

### Descarga de métricas faltantes

La comparación realizada mostró que las descargas realizadas mediante filtros temáticos incluían 46 de las 130 métricas del Fashion Transparency Index 2023.

Para recuperar las 84 métricas restantes, estas se dividieron en cinco bloques de descarga. Los primeros cuatro bloques contienen 20 métricas cada uno, equivalentes a 5,000 observaciones por archivo, mientras que el último bloque contiene 4 métricas, equivalentes a 1,000 observaciones.

Los cinco archivos fueron almacenados en la carpeta `raw/Other` para posteriormente verificar su estructura e integrarlos con las descargas temáticas.

In [11]:
# Definimos la carpeta donde se encuentran los bloques adicionales

ruta_other = Path("../raw/Other")

# Buscamos todos los archivos CSV
archivos_other = sorted(ruta_other.glob("*.csv"))

# Mostramos cuántos archivos se encontraron
print(f"Se encontraron {len(archivos_other)} archivos CSV.\n")

# Mostramos el nombre de cada archivo
for archivo in archivos_other:
    print(archivo.name)

Se encontraron 5 archivos CSV.

Bloque_1.csv
Bloque_2.csv
Bloque_3.csv
Bloque_4.csv
Bloque_5.csv


In [12]:
# Revisamos la estructura de los archivos adicionales

estructura_other = []

# Recorremos cada archivo
for archivo in archivos_other:

    # Leemos el archivo
    df = pd.read_csv(archivo, skiprows=4)

    # Guardamos un resumen de su estructura
    estructura_other.append({
        "archivo": archivo.name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "variables": tuple(df.columns)
    })

# Convertimos el resumen en un DataFrame
estructura_other = pd.DataFrame(estructura_other)

# Mostramos el resumen
estructura_other[["archivo", "filas", "columnas"]]

,archivo,filas,columnas
0,Bloque_1.csv,5000,6
1,Bloque_2.csv,5000,6
2,Bloque_3.csv,5000,6
3,Bloque_4.csv,5000,6
4,Bloque_5.csv,1000,6


In [13]:
# Comparamos las columnas presentes en cada archivo adicional

columnas_other = pd.DataFrame({
    archivo.name: pd.read_csv(archivo, skiprows=4).columns
    for archivo in archivos_other
})

columnas_other

,Bloque_1.csv,Bloque_2.csv,Bloque_3.csv,Bloque_4.csv,Bloque_5.csv
0,Answer Page,Answer Page,Answer Page,Answer Page,Answer Page
1,Metric,Metric,Metric,Metric,Metric
2,Company,Company,Company,Company,Company
3,Year,Year,Year,Year,Year
4,Value,Value,Value,Value,Value
5,Source Page,Source Page,Source Page,Source Page,Source Page


In [14]:
# Verificamos si las columnas de los archivos adicionales son un subconjunto
# de las columnas de la base anterior

# Obtenemos las columnas de la base anterior
# sin considerar la columna Categoria, que fue agregada durante la integración
columnas_originales = set(base_integrada.columns) - {"Categoria"}

# Obtenemos las columnas de un archivo adicional
columnas_adicionales = set(
    pd.read_csv(archivos_other[0], skiprows=4).columns
)

# Mostramos ambos conjuntos de columnas
print("Columnas originales:", columnas_originales)
print("\nColumnas adicionales:", columnas_adicionales)

# Verificamos si todas las columnas adicionales existen en la base anterior
print(
    "\n¿Las columnas adicionales están contenidas en las originales?",
    columnas_adicionales.issubset(columnas_originales)
)

Columnas originales: {'Metric', 'Original Source', 'Year', 'Answer ID', 'ISIN', 'Comments', 'Company', 'Source Page', 'Source Count', 'Value', 'Answer Page'}

Columnas adicionales: {'Metric', 'Year', 'Company', 'Source Page', 'Value', 'Answer Page'}

¿Las columnas adicionales están contenidas en las originales? True


In [15]:
# Igualamos la estructura de los archivos adicionales

# Guardamos el orden de columnas de la base anterior
columnas_base = base_integrada.columns.tolist()

# Creamos una lista para almacenar los DataFrames
lista_other = []

# Recorremos cada archivo adicional
for archivo in archivos_other:

    # Leemos el archivo
    df = pd.read_csv(archivo, skiprows=4)

    # Registramos el origen del archivo
    df["Categoria"] = archivo.parent.name

    # Agregamos las columnas faltantes
    for columna in columnas_base:

        # Si la columna no existe, la creamos con valores vacíos
        if columna not in df.columns:
            df[columna] = pd.NA

    # Reordenamos las columnas para que coincidan con la base anterior
    df = df[columnas_base]

    # Guardamos el DataFrame
    lista_other.append(df)

In [16]:
# Integramos los archivos adicionales en una sola base

base_other = pd.concat(
    lista_other,
    ignore_index=True
)

# Verificamos el tamaño de la base resultante
print(f"Número total de registros: {len(base_other):,}")
print(f"Número de variables: {base_other.shape[1]}")

Número total de registros: 21,000
Número de variables: 12


In [17]:
# Verificamos registros únicos y duplicados en ambas bases

clave = ["Company", "Metric", "Year"]

print("Base temática")
print("Registros totales:", len(base_integrada))
print("Registros únicos:", len(base_integrada.drop_duplicates(subset=clave)))
print("Duplicados:", base_integrada.duplicated(subset=clave).sum())

print("\nMétricas faltantes")
print("Registros totales:", len(base_other))
print("Registros únicos:", len(base_other.drop_duplicates(subset=clave)))
print("Duplicados:", base_other.duplicated(subset=clave).sum())

Base temática
Registros totales: 27000
Registros únicos: 11500
Duplicados: 15500

Métricas faltantes
Registros totales: 21000
Registros únicos: 21000
Duplicados: 0


### Consolidación de la base de datos

Una vez homologada la estructura de los archivos adicionales, ambas bases fueron integradas en una sola tabla. Posteriormente, se eliminaron los registros duplicados utilizando como unidad de análisis la combinación **empresa – indicador – año**, obteniendo la versión consolidada del Fashion Transparency Index 2023.

In [18]:
# Integramos la base anterior con la base de métricas faltantes

base_final = pd.concat([base_integrada, base_other], ignore_index=True)

# Eliminamos registros duplicados utilizando la unidad de análisis
base_final = base_final.drop_duplicates(
    subset=["Company", "Metric", "Year"]
).reset_index(drop=True)

# Mostramos el tamaño de la base final
print(f"Registros finales: {len(base_final):,}")
print(f"Variables: {base_final.shape[1]}")

Registros finales: 32,500
Variables: 12


/var/folders/qj/3qylzbbx2r943y2gxcy3t9vc0000gn/T/ipykernel_36773/2015599239.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  base_final = pd.concat([base_integrada, base_other], ignore_index=True)


### Diccionarios de variables e indicadores

Como parte del proceso de construcción de la base de datos, se elaboraron dos diccionarios complementarios. El primero documenta el significado y el tipo de dato de las variables que conforman la base consolidada. El segundo describe cada indicador del Fashion Transparency Index 2023, su categoría temática y el tipo de respuesta observado.

Estos diccionarios facilitan la interpretación de la información y servirán como referencia para las etapas posteriores de limpieza y análisis.

In [19]:
# Revisamos las variables que conforman la base final

base_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32500 entries, 0 to 32499
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Answer Page      32500 non-null  object 
 1   Metric           32500 non-null  object 
 2   Company          32500 non-null  object 
 3   Year             32500 non-null  int64  
 4   Value            31244 non-null  object 
 5   Source Page      31000 non-null  object 
 6   Answer ID        10500 non-null  float64
 7   Original Source  10500 non-null  object 
 8   Source Count     10500 non-null  float64
 9   Comments         1998 non-null   object 
 10  ISIN             2714 non-null   object 
 11  Categoria        32500 non-null  object 
dtypes: float64(2), int64(1), object(9)
memory usage: 3.0+ MB


In [20]:
# Construimos el diccionario de variables

diccionario_variables = pd.DataFrame({
    "Variable": base_final.columns,
    "Tipo de dato observado": base_final.dtypes.astype(str).values
})

# Traducimos los tipos de dato para facilitar su interpretación

diccionario_variables["Tipo de dato observado"] = (
    diccionario_variables["Tipo de dato observado"]
    .replace({
        "object": "Texto",
        "int64": "Entero",
        "float64": "Numérico decimal"
    })
)

# Agregamos la descripción de cada variable

diccionario_variables["Descripción"] = [
    "Enlace de WikiRate correspondiente al registro de la respuesta.",
    "Indicador del Fashion Transparency Index 2023 evaluado para la empresa.",
    "Nombre de la empresa evaluada.",
    "Año al que corresponde la evaluación.",
    "Valor o respuesta reportada para el indicador.",
    "Enlace a la página de WikiRate donde se documenta la fuente de la respuesta.",
    "Identificador único de la respuesta dentro de WikiRate.",
    "Documento o fuente original utilizada como evidencia para la respuesta.",
    "Número de fuentes asociadas con la respuesta.",
    "Comentarios adicionales registrados en WikiRate.",
    "Código ISIN (International Securities Identification Number) de la empresa, cuando se encuentra disponible.",
    "Categoría temática asociada con el archivo de origen del indicador."
]

# Reordenamos las columnas

diccionario_variables = diccionario_variables[
    ["Variable", "Descripción", "Tipo de dato observado"]
]

# Mostramos el diccionario de variables

diccionario_variables

,Variable,Descripción,Tipo de dato observado
0,Answer Page,Enlace de WikiRate correspondiente al registro...,Texto
1,Metric,Indicador del Fashion Transparency Index 2023 ...,Texto
2,Company,Nombre de la empresa evaluada.,Texto
3,Year,Año al que corresponde la evaluación.,Entero
4,Value,Valor o respuesta reportada para el indicador.,Texto
5,Source Page,Enlace a la página de WikiRate donde se docume...,Texto
6,Answer ID,Identificador único de la respuesta dentro de ...,Numérico decimal
7,Original Source,Documento o fuente original utilizada como evi...,Texto
8,Source Count,Número de fuentes asociadas con la respuesta.,Numérico decimal
9,Comments,Comentarios adicionales registrados en WikiRate.,Texto


In [21]:
# Obtenemos la lista única de indicadores de la base final

diccionario_indicadores = pd.DataFrame({
    "Indicador original": sorted(base_final["Metric"].dropna().unique())
})

# Generamos una versión más legible del nombre de cada indicador
diccionario_indicadores["Indicador"] = (
    diccionario_indicadores["Indicador original"]
    .str.replace("Fashion Revolution+", "", regex=False)
)

# Mostramos el número total de indicadores identificados
print(f"Indicadores identificados: {len(diccionario_indicadores)}")

# Mostramos las primeras filas del diccionario
diccionario_indicadores.head()

Indicadores identificados: 130


,Indicador original,Indicador
0,Fashion Revolution+1. Policy & Commitments Score,1. Policy & Commitments Score
1,Fashion Revolution+1.1 Own Operations Policies,1.1 Own Operations Policies
2,Fashion Revolution+1.3 Management Procedures,1.3 Management Procedures
3,Fashion Revolution+1.5 Verified Sustainability...,1.5 Verified Sustainability Report
4,Fashion Revolution+2. Governance Score,2. Governance Score


In [22]:
# Obtenemos la categoría temática de los indicadores
# presentes en las descargas originales

categorias_originales = (
    base_final[
        base_final["Categoria"] != "Other"
    ][["Metric", "Categoria"]]
    .drop_duplicates()
)

# Eliminamos el prefijo utilizado por WikiRate

categorias_originales["Metric"] = (
    categorias_originales["Metric"]
    .str.replace("Fashion Revolution+", "", regex=False)
)

# Incorporamos la categoría al diccionario de indicadores

diccionario_indicadores = diccionario_indicadores.merge(
    categorias_originales,
    left_on="Indicador",
    right_on="Metric",
    how="left"
)

# Eliminamos la columna auxiliar

diccionario_indicadores = (
    diccionario_indicadores
    .drop(columns="Metric")
    .rename(columns={"Categoria": "Categoría temática"})
)

# Revisamos cuántos indicadores ya fueron clasificados automáticamente

diccionario_indicadores["Categoría temática"].value_counts(dropna=False)

Categoría temática
NaN            84
Environment    26
Governance     12
Social          8
Name: count, dtype: int64

In [23]:
# Clasificación manual de los indicadores descargados desde Other

categorias_manuales = {
    
    # Governance

    "Supply Chain Policies": "Governance",
    "Supply Chain Policies Align with International Standards": "Governance",
    "Supply Chain Policies Are Contractual": "Governance",
    "Supply Chain Policies in Local Language": "Governance",
    "1. Policy & Commitments Score": "Governance",
    "1.1 Own Operations Policies": "Governance",
    "1.3 Management Procedures": "Governance",
    "1.5 Verified Sustainability Report": "Governance",
    "2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues": "Governance",
    "Accountable Board Member Identified": "Governance",
    "Implementation of Board Level Accountability Described": "Governance",
    "Employee Incentives to Improve Impacts": "Governance",
    "Executive Incentives to Improve Impacts": "Governance",
    "Supplier Incentives to Improve Impacts": "Governance",
    "Responsible Tax Strategy": "Governance",

    # Traceability

    "3. Traceability Score (2023)": "Traceability",
    "3.1 Tier One Factory Disclosure": "Traceability",
    "3.2 Processing Facilities Disclosure": "Traceability",
    "3.3 Raw Materials Suppliers Disclosure": "Traceability",

    # Social

    "Plan for Improving Human Rights Impacts": "Social",
    "Reports on Efforts to Improve Human Rights Impacts": "Social",
    "Describes Human Rights Due Diligence Process": "Social",
    "Approach to Involving Women in Human Rights Due Diligence": "Social",
    "Human Rights Risks Impacts and Violations Identified": "Social",
    "Prevention Mitigation and Remediation of Human Rights Risks": "Social",
    "Human Rights Risk Prevention and Remediation Outcomes Published": "Social",
    "New Production Facility Criteria": "Social",
    "Number or % of off-site worker interviews": "Social",
    "Percentage of Audits including Trade Union Representative": "Social",
    "Summary of Assessment Findings": "Social",
    "Ratings by Named Facilities": "Social",
    "Selected Audit Findings by Named Facilities": "Social",
    "Full Audit Reports by Named Facilities": "Social",
    "Remediation Process": "Social",
    "Affected Stakeholder Engagement in Remediation": "Social",
    "Exit Strategy": "Social",
    "Grievance Mechanism - Direct Employees": "Social",
    "Grievance Mechanism - Supply Chain Workers": "Social",
    "Grievance Mechanism Implementation - Supply Chain Workers": "Social",
    "Grievance Mechanism Disseminated to Supply Chain Workers": "Social",
    "Grievance Mechanism in Supplier Policies": "Social",
    "Grievance Reporting - Supply Chain Workers": "Social",
    "Discloses Approach to Recruitment Fees": "Social",
    "Discloses Data on Modern Slavery Prevalence": "Social",
    "Discloses Approach to Living Wage": "Social",
    "Discloses Strategy to Achieving Living Wage": "Social",
    "Discloses Progress toward the payment of a Living Wage to workers in the supply chain": "Social",
    "Discloses Living Wage Estimates Used for Benchmarking": "Social",
    "Discloses Percentage of Workers Receiving Wage Payments Digitally": "Social",
    "Publishes Percentage of Workers Paid Above Minimum Wage": "Social",
    "Publishes Percentage or Number of Workers Earning a Living Wage": "Social",
    "Protects Labour Costs in Price Negotiations": "Social",
    "Discloses Quantity of Orders with Labor Cost Protection": "Social",
    "Policy to Pay Supplier Within 60 Days": "Social",
    "Discloses Quantity of Orders Changed After Original Agreement": "Social",
    "Publishes Supplier Feedback on Purchasing Practices": "Social",
    "Discloses number or % of supplier facilities that have independent, democratically elected trade unions": "Social",
    "Discloses number or % of Workers covered by Collective Bargaining Agreements": "Social",
    "Publishes Gender Pay Gap": "Social",
    "Publishes Sex-disaggregated Job Distribution": "Social",
    "Publishes Gender-based Labour Violations Data": "Social",
    "Discloses Gender Equality Actions in Supplier Facilities": "Social",
    "Publishes Ethnicity Pay Gap": "Social",
    "Publishes Race-disaggregated Job Distribution": "Social",

    # Environment

    "Plan for Improving Environmental Impacts": "Environment",
    "Reports on Efforts to Improve Environmental Impacts": "Environment",
    "Sustainable Materials Strategy": "Environment",
    "Discloses Progress on Sustainable Materials Strategy": "Environment",
    "Targets to Reduce Virgin Plastics": "Environment",
    "Discloses Progress to Reducing Virgin Plastics": "Environment",
    "Minimizing Impact of Microfibres": "Environment",
    "Discloses Quantity of Products Produced": "Environment",
    "Commitment to Degrowth": "Environment",
    "Discloses Quantity of Products Destroyed": "Environment",
    "Offers Take-back Schemes": "Environment",
    "Offers Repair Services": "Environment",
    "Discloses Evidence of Developing Circular Solutions": "Environment",
    "Discloses % of Circular Products": "Environment",
    "Commitment to Eliminate Hazardous Chemicals": "Environment",
    "Discloses Progress to Eliminate Hazardous Chemicals": "Environment",
    "Science Based Targets": "Environment",
    "Environmental Profit and Loss Statement": "Environment",
    "Zero Deforestation Commitment": "Environment",
    "Implementation of Regenerative Farming Practices": "Environment",
    "Discloses Renewable Energy Use": "Environment"
}

In [24]:
# Completamos las categorías faltantes

diccionario_indicadores["Categoría temática"] = (
    diccionario_indicadores["Categoría temática"]
    .fillna(
        diccionario_indicadores["Indicador"].map(categorias_manuales)
    )
)

In [25]:
# Verificamos que todos los indicadores tengan categoría temática

print(
    diccionario_indicadores["Categoría temática"]
    .value_counts(dropna=False)
)

print(
    "\nIndicadores sin categoría:",
    diccionario_indicadores["Categoría temática"].isna().sum()
)

Categoría temática
Social          53
Environment     47
Governance      26
Traceability     4
Name: count, dtype: int64

Indicadores sin categoría: 0


In [26]:
# Identificamos el tipo de respuesta observado para cada indicador

tipos_respuesta = []

for indicador in diccionario_indicadores["Indicador original"]:

    # Valores observados para el indicador
    valores = (
        base_final.loc[
            base_final["Metric"] == indicador,
            "Value"
        ]
        .dropna()
        .astype(str)
    )

    # Clasificamos automáticamente el tipo de respuesta
    if valores.isin(["Yes", "No"]).all():
        tipo = "Sí / No"

    elif valores.str.fullmatch(r"-?\d+(\.\d+)?").all():
        tipo = "Numérico"

    elif valores.str.contains("%").any():
        tipo = "Porcentaje"

    elif valores.str.contains(",").any():
        tipo = "Lista de elementos"

    else:
        tipo = "Texto"

    tipos_respuesta.append(tipo)

# Agregamos la columna al diccionario

diccionario_indicadores["Tipo de respuesta"] = tipos_respuesta

In [27]:
# Imprimimos las primeras 20 filas

diccionario_indicadores[
    ["Indicador", "Tipo de respuesta"]
].head(20)

,Indicador,Tipo de respuesta
0,1. Policy & Commitments Score,Numérico
1,1.1 Own Operations Policies,Lista de elementos
2,1.3 Management Procedures,Lista de elementos
3,1.5 Verified Sustainability Report,Sí / No
4,2. Governance Score,Numérico
5,2.1 Identifies Lead Responsibility for Human R...,Texto
6,3. Traceability Score (2023),Numérico
7,3.1 Tier One Factory Disclosure,Porcentaje
8,3.2 Processing Facilities Disclosure,Porcentaje
9,3.3 Raw Materials Suppliers Disclosure,Porcentaje


In [28]:
# Revisamos cuántos indicadores hay por tipo de respuesta

diccionario_indicadores["Tipo de respuesta"].value_counts()

Tipo de respuesta
Sí / No               110
Lista de elementos     10
Numérico                6
Porcentaje              3
Texto                   1
Name: count, dtype: int64

In [29]:
# Revisamos los indicadores clasificados como "Texto"

diccionario_indicadores.loc[
    diccionario_indicadores["Tipo de respuesta"] == "Texto",
    ["Indicador", "Tipo de respuesta"]
]

,Indicador,Tipo de respuesta
5,2.1 Identifies Lead Responsibility for Human R...,Texto


In [30]:
# Identificamos los indicadores numéricos que corresponden a puntajes

indicadores_puntaje = [
    "1. Policy & Commitments Score",
    "2. Governance Score",
    "3. Traceability Score (2023)",
    "4. Know, Show & Fix Score",
    "5. Spotlight Issues Score (2023)",
    "Fashion Transparency Index 2023"
]

# Reemplazamos su tipo de respuesta

diccionario_indicadores.loc[
    diccionario_indicadores["Indicador"].isin(indicadores_puntaje),
    "Tipo de respuesta"
] = "Puntaje"

In [31]:
# Agregamos una breve descripción de cada indicador

significados = {

    "1. Policy & Commitments Score": "Puntaje de transparencia sobre políticas y compromisos de sostenibilidad.",
    "1.1 Own Operations Policies": "Políticas aplicables a las operaciones propias de la empresa.",
    "1.3 Management Procedures": "Procedimientos de gestión para implementar las políticas de sostenibilidad.",
    "1.5 Verified Sustainability Report": "Publicación de un informe de sostenibilidad verificado externamente.",
    "2. Governance Score": "Puntaje de transparencia sobre gobernanza y rendición de cuentas.",
    "2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues": "Persona o área responsable de los temas de derechos humanos y medio ambiente.",
    "3. Traceability Score (2023)": "Puntaje de transparencia sobre trazabilidad en la cadena de suministro.",
    "3.1 Tier One Factory Disclosure": "Divulgación de las fábricas de primer nivel de la cadena de suministro.",
    "3.2 Processing Facilities Disclosure": "Divulgación de las instalaciones donde se procesan los materiales.",
    "3.3 Raw Materials Suppliers Disclosure": "Divulgación de los proveedores de materias primas.",
    "4. Know, Show & Fix Score": "Puntaje sobre identificación, divulgación y corrección de impactos.",
    "5. Spotlight Issues Score (2023)": "Puntaje sobre transparencia en temas prioritarios de sostenibilidad.",
    "Accountable Board Member Identified": "Identificación de un miembro del consejo responsable de sostenibilidad.",
    "Affected Stakeholder Engagement in Remediation": "Participación de las partes afectadas en los procesos de remediación.",
    "Approach to Defining Sustainable Materials": "Criterios utilizados para definir materiales sostenibles.",
    "Approach to Involving Women in Human Rights Due Diligence": "Acciones para incluir a las mujeres en la debida diligencia de derechos humanos.",
    "Commitment to Degrowth": "Compromiso con estrategias de reducción del crecimiento productivo.",
    "Commitment to Eliminate Hazardous Chemicals": "Compromiso para eliminar sustancias químicas peligrosas.",
    "Decarbonisation Commitment": "Compromiso para reducir las emisiones de carbono.",
    "Decarbonisation Progress": "Reporte de avances en las metas de descarbonización.",
    "Describes Environmental Due Diligence Process": "Descripción del proceso de debida diligencia ambiental.",
    "Describes Human Rights Due Diligence Process": "Descripción del proceso de debida diligencia en derechos humanos.",
    "Discloses % of Circular Products": "Divulgación del porcentaje de productos circulares.",
    "Discloses Absolute Energy Reduction": "Divulgación de la reducción absoluta en el consumo de energía.",
    "Discloses Annual Investment in Decarbonisation": "Divulgación de la inversión anual destinada a la descarbonización.",
    "Discloses Approach to Living Wage": "Divulgación del enfoque para promover salarios dignos.",
    "Discloses Approach to Recruitment Fees": "Divulgación del enfoque para gestionar las cuotas de contratación.",
    "Discloses Breakdown of Reuse/Recyling of Pre-consumer Waste": "Divulgación del destino de los residuos previos al consumo.",
    "Discloses Coal Use": "Divulgación del uso de carbón en las operaciones.",
    "Discloses Content of Scope 1, 2 and 3 Emissions": "Divulgación de emisiones de alcance 1, 2 y 3.",
    "Discloses Data on Modern Slavery Prevalence": "Divulgación de información sobre esclavitud moderna.",
    "Discloses Efforts to Invest in Supply Chain Workers": "Divulgación de inversiones dirigidas a trabajadores de la cadena de suministro.",
    "Discloses Evidence of Developing Circular Solutions": "Divulgación de acciones para desarrollar soluciones circulares.",
    "Discloses Free on Board (FOB) Price Changes (COVID-19 Response)": "Divulgación de cambios en precios FOB durante la respuesta a COVID-19.",
    "Discloses Gender Equality Actions in Supplier Facilities": "Divulgación de acciones para promover la igualdad de género en proveedores.",
    "Discloses Living Wage Estimates Used for Benchmarking": "Divulgación de estimaciones de salario digno utilizadas como referencia.",
    "Discloses Number of Collective Bargaining Agreements Providing Wages Above Legal Minimum": "Divulgación de convenios colectivos con salarios superiores al mínimo legal.",
    "Discloses Number of Workers Affected by Recruitment Fees": "Divulgación del número de trabajadores afectados por cuotas de contratación.",
    "Discloses Percentage of Workers Paid By Piece Rate": "Divulgación del porcentaje de trabajadores remunerados por destajo.",
    "Discloses Percentage of Workers Receiving Wage Payments Digitally": "Divulgación del porcentaje de trabajadores que reciben pagos digitales.",
    "Discloses Policy on Up-Front Supplier Payments": "Divulgación de la política de pagos anticipados a proveedores.",
    "Discloses Prevalance of Collective Bargaining Violations": "Divulgación de casos de incumplimiento de negociación colectiva.",
    "Discloses Progress on Sustainable Materials Strategy": "Divulgación de avances en la estrategia de materiales sostenibles.",
    "Discloses Progress to Eliminate Hazardous Chemicals": "Divulgación de avances para eliminar sustancias químicas peligrosas.",
    "Discloses Progress to Reducing Textiles Derived from Virgin Fossil Fuels": "Divulgación de avances para reducir textiles derivados de combustibles fósiles vírgenes.",
    "Discloses Progress to Reducing Virgin Plastics": "Divulgación de avances para reducir el uso de plásticos vírgenes.",
    "Discloses Progress toward the payment of a Living Wage to workers in the supply chain": "Divulgación de avances en el pago de salarios dignos en la cadena de suministro.",
    "Discloses Proportion of Workers Paid Minimum Wage": "Divulgación de la proporción de trabajadores que reciben el salario mínimo.",
    "Discloses Quantity of Orders Changed After Original Agreement": "Divulgación de la cantidad de pedidos modificados después del acuerdo inicial.",
    "Discloses Quantity of Orders with Labor Cost Protection": "Divulgación de pedidos que protegen los costos laborales.",
    "Discloses Quantity of Post-production Waste Generated": "Divulgación de la cantidad de residuos generados después de la producción.",
    "Discloses Quantity of Pre-Production Waste Generated": "Divulgación de la cantidad de residuos generados antes de la producción.",
    "Discloses Quantity of Products Destroyed": "Divulgación de la cantidad de productos destruidos.",
    "Discloses Quantity of Products Produced": "Divulgación de la cantidad de productos fabricados.",
    "Discloses Renewable Energy Use": "Divulgación del uso de energías renovables.",
    "Discloses Sourced Fibre Breakdown": "Divulgación de la composición de las fibras utilizadas.",
    "Discloses Strategy to Achieving Living Wage": "Divulgación de la estrategia para alcanzar salarios dignos.",
    "Discloses Take-back Scheme Outcomes": "Divulgación de los resultados de los programas de devolución de productos.",
    "Discloses Time Taken to Pay Purchase Orders": "Divulgación del tiempo requerido para pagar órdenes de compra.",
    "Discloses Water Use": "Divulgación del consumo de agua.",
    "Discloses Water-Related Risk Assessment Process": "Divulgación del proceso de evaluación de riesgos relacionados con el agua.",
    "Discloses number or % of Workers covered by Collective Bargaining Agreements": "Divulgación del número o porcentaje de trabajadores cubiertos por convenios colectivos.",
    "Discloses number or % of supplier facilities that have independent, democratically elected trade unions": "Divulgación del número o porcentaje de proveedores con sindicatos independientes.",
    "Employee Incentives to Improve Impacts": "Incentivos para que los empleados mejoren los impactos de sostenibilidad.",
    "Environmental Profit and Loss Statement": "Publicación de un estado de ganancias y pérdidas ambientales.",
    "Environmental Risk Prevention and Remediation Outcomes Published": "Publicación de resultados sobre prevención y remediación de riesgos ambientales.",
    "Environmental Risks Impacts and Violations Identified": "Identificación de riesgos, impactos e incumplimientos ambientales.",
    "Executive Incentives to Improve Impacts": "Incentivos para directivos vinculados a mejoras en sostenibilidad.",
    "Executive Pay Linked to Environmental and Social Targets": "Vinculación de la remuneración directiva con objetivos ambientales y sociales.",
    "Exit Strategy": "Divulgación de la estrategia para finalizar relaciones con proveedores.",
    "Fashion Transparency Index 2023": "Puntaje general obtenido en el Fashion Transparency Index 2023.",
    "Full Audit Reports by Named Facilities": "Publicación de informes completos de auditoría por instalación.",
    "Grievance Mechanism - Direct Employees": "Existencia de mecanismos de quejas para empleados directos.",
    "Grievance Mechanism - Supply Chain Workers": "Existencia de mecanismos de quejas para trabajadores de la cadena de suministro.",
    "Grievance Mechanism Disseminated to Supply Chain Workers": "Difusión de los mecanismos de quejas entre trabajadores de la cadena de suministro.",
    "Grievance Mechanism Implementation - Supply Chain Workers": "Implementación de mecanismos de quejas para trabajadores de la cadena de suministro.",
    "Grievance Mechanism in Supplier Policies": "Inclusión de mecanismos de quejas en las políticas para proveedores.",
    "Grievance Reporting - Supply Chain Workers": "Divulgación de reportes derivados de mecanismos de quejas.",
    "Human Rights Risk Prevention and Remediation Outcomes Published": "Publicación de resultados sobre prevención y remediación de riesgos en derechos humanos.",
    "Human Rights Risks Impacts and Violations Identified": "Identificación de riesgos, impactos e incumplimientos en derechos humanos.",
    "Implementation of Board Level Accountability Described": "Descripción de cómo el consejo asume responsabilidades en sostenibilidad.",
    "Implementation of Regenerative Farming Practices": "Divulgación de prácticas de agricultura regenerativa.",
    "Minimizing Impact of Microfibres": "Acciones para reducir el impacto de las microfibras.",
    "New Production Facility Criteria": "Criterios utilizados para seleccionar nuevas instalaciones de producción.",
    "Number or % of off-site worker interviews": "Divulgación del número o porcentaje de entrevistas realizadas fuera del lugar de trabajo.",
    "Offers Clothing Longevity Business Models": "Oferta de modelos de negocio que prolongan la vida útil de las prendas.",
    "Offers Repair Services": "Oferta de servicios de reparación de prendas.",
    "Offers Take-back Schemes": "Oferta de programas para recuperar productos usados.",
    "Percentage of Audits including Trade Union Representative": "Divulgación del porcentaje de auditorías con participación sindical.",
    "Plan for Improving Environmental Impacts": "Plan para reducir los impactos ambientales.",
    "Reports on Efforts to Improve Environmental Impacts": "Reporte de acciones para mejorar los impactos ambientales.",
    "Reports on Efforts to Improve Human Rights Impacts": "Reporte de acciones para mejorar los impactos en derechos humanos.",
    "Reports on Minimum Wage Paid for Daily / Piece Rate Workers": "Reporte sobre el salario mínimo pagado a trabajadores por día o destajo.",
    "Responsible Tax Strategy": "Divulgación de una estrategia fiscal responsable.",
    "Science Based Targets": "Adopción de metas climáticas alineadas con la ciencia.",
    "Scope, Process and Accreditation for Environmental Audits": "Divulgación del alcance, proceso y acreditación de las auditorías ambientales.",
    "Selected Audit Findings by Named Facilities": "Divulgación de hallazgos de auditorías por instalación.",
    "Stakeholder Engagement in Environmental Due Diligence": "Participación de las partes interesadas en la debida diligencia ambiental.",
    "Stakeholder Engagement in Human Rights Due Diligence": "Participación de las partes interesadas en la debida diligencia en derechos humanos.",
    "Summary of Assessment Findings": "Resumen de los resultados de las evaluaciones realizadas.",
    "Supplier Incentives to Improve Impacts": "Incentivos para que los proveedores mejoren sus impactos.",
    "Supply Chain Policies": "Políticas aplicables a la cadena de suministro.",
    "Supply Chain Policies Align with International Standards": "Alineación de las políticas con estándares internacionales.",
    "Supply Chain Policies Are Contractual": "Incorporación de las políticas en contratos con proveedores.",
    "Supply Chain Policies in Local Language": "Disponibilidad de las políticas en el idioma local.",
    "Sustainable Materials Strategy": "Estrategia para incrementar el uso de materiales sostenibles.",
    "Targets to Reduce Textiles Derived from Virgin Fossil Fuels": "Metas para reducir textiles derivados de combustibles fósiles vírgenes.",
    "Targets to Reduce Virgin Plastics": "Metas para reducir el uso de plásticos vírgenes.",
    "Worker Representation on Board": "Representación de los trabajadores en el consejo de administración.",
    "Zero Deforestation Commitment": "Compromiso para eliminar la deforestación en la cadena de suministro.",
    "Zero Deforestation Progress": "Reporte de avances hacia la eliminación de la deforestación.",
    "Plan for Improving Human Rights Impacts": "Plan para reducir los impactos en derechos humanos.",
    "Policy to Pay Supplier Within 60 Days": "Política para pagar a los proveedores en un plazo máximo de 60 días.",
    "Prevention Mitigation and Remediation of Environmental Risks": "Acciones para prevenir, mitigar y remediar riesgos ambientales.",
    "Prevention Mitigation and Remediation of Human Rights Risks": "Acciones para prevenir, mitigar y remediar riesgos en derechos humanos.",
    "Protects Labour Costs in Price Negotiations": "Protección de los costos laborales durante la negociación de precios.",
    "Publishes Ethnicity Pay Gap": "Publicación de la brecha salarial por origen étnico.",
    "Publishes Gender Pay Gap": "Publicación de la brecha salarial de género.",
    "Publishes Gender-based Labour Violations Data": "Publicación de datos sobre violaciones laborales por motivo de género.",
    "Publishes Percentage of Workers Paid Above Minimum Wage": "Publicación del porcentaje de trabajadores que reciben un salario superior al mínimo.",
    "Publishes Percentage or Number of Workers Earning a Living Wage": "Publicación del número o porcentaje de trabajadores que reciben un salario digno.",
    "Publishes Race-disaggregated Job Distribution": "Publicación de la distribución de puestos desagregada por raza.",
    "Publishes Racial Equality Actions": "Publicación de acciones para promover la igualdad racial.",
    "Publishes Responsible Purchasing Code of Conduct": "Publicación del código de conducta para compras responsables.",
    "Publishes Sex-disaggregated Job Distribution": "Publicación de la distribución de puestos desagregada por sexo.",
    "Publishes Standard Supplier Agreement Template": "Publicación del modelo estándar de contrato con proveedores.",
    "Publishes Supplier Feedback on Purchasing Practices": "Publicación de la retroalimentación de proveedores sobre las prácticas de compra.",
    "Publishes Supplier Wastewater Test Results": "Publicación de resultados de pruebas de aguas residuales de proveedores.",
    "Ratings by Named Facilities": "Publicación de calificaciones por instalación.",
    "Remediation Process": "Descripción del proceso de remediación aplicado por la empresa."
}

# Incorporamos el significado al diccionario
diccionario_indicadores["Significado"] = (
    diccionario_indicadores["Indicador"]
    .map(significados)
)

# Reordenamos las columnas
diccionario_indicadores = diccionario_indicadores[
    [
        "Indicador original",
        "Indicador",
        "Significado",
        "Categoría temática",
        "Tipo de respuesta"
    ]
]

# Mostramos el diccionario
diccionario_indicadores

,Indicador original,Indicador,Significado,Categoría temática,Tipo de respuesta
0,Fashion Revolution+1. Policy & Commitments Score,1. Policy & Commitments Score,Puntaje de transparencia sobre políticas y com...,Governance,Puntaje
1,Fashion Revolution+1.1 Own Operations Policies,1.1 Own Operations Policies,Políticas aplicables a las operaciones propias...,Governance,Lista de elementos
2,Fashion Revolution+1.3 Management Procedures,1.3 Management Procedures,Procedimientos de gestión para implementar las...,Governance,Lista de elementos
3,Fashion Revolution+1.5 Verified Sustainability...,1.5 Verified Sustainability Report,Publicación de un informe de sostenibilidad ve...,Governance,Sí / No
4,Fashion Revolution+2. Governance Score,2. Governance Score,Puntaje de transparencia sobre gobernanza y re...,Governance,Puntaje
...,...,...,...,...,...
125,Fashion Revolution+Targets to Reduce Textiles ...,Targets to Reduce Textiles Derived from Virgin...,Metas para reducir textiles derivados de combu...,Environment,Sí / No
126,Fashion Revolution+Targets to Reduce Virgin Pl...,Targets to Reduce Virgin Plastics,Metas para reducir el uso de plásticos vírgenes.,Environment,Sí / No
127,Fashion Revolution+Worker Representation on Board,Worker Representation on Board,Representación de los trabajadores en el conse...,Governance,Sí / No
128,Fashion Revolution+Zero Deforestation Commitment,Zero Deforestation Commitment,Compromiso para eliminar la deforestación en l...,Environment,Sí / No


In [32]:
# Verificamos que todos los indicadores tengan significado

print("Indicadores definidos:", len(significados))
print("Significados faltantes:", diccionario_indicadores["Significado"].isna().sum())

Indicadores definidos: 130
Significados faltantes: 0


### Validación final de la base de datos y los diccionarios

In [33]:
# VALIDACIÓN FINAL DE LA BASE DE DATOS

# Mostramos el número de registros y variables
print("Número de registros:", len(base_final))
print("Número de variables:", base_final.shape[1])


# Verificamos si existen valores faltantes
print("\nValores faltantes por variable:")

print(
    base_final
    .isna()
    .sum()
)


# Verificamos si existen registros duplicados
print("\nRegistros duplicados:")

print(
    base_final
    .duplicated()
    .sum()
)


# Mostramos el tipo de dato de cada variable
print("\nTipo de dato de cada variable:")

print(
    base_final
    .dtypes
)

Número de registros: 32500
Número de variables: 12

Valores faltantes por variable:
Answer Page            0
Metric                 0
Company                0
Year                   0
Value               1256
Source Page         1500
Answer ID          22000
Original Source    22000
Source Count       22000
Comments           30502
ISIN               29786
Categoria              0
dtype: int64

Registros duplicados:
0

Tipo de dato de cada variable:
Answer Page         object
Metric              object
Company             object
Year                 int64
Value               object
Source Page         object
Answer ID          float64
Original Source     object
Source Count       float64
Comments            object
ISIN                object
Categoria           object
dtype: object


**Observación**

Los valores faltantes identificados corresponden a variables cuyo contenido no está disponible para todos los registros en la fuente original de Fashion Revolution (por ejemplo, *Comments*, *ISIN*, *Answer ID*, *Original Source* y *Source Count*). Asimismo, la variable **Categoria** únicamente se asignó a los indicadores provenientes de las categorías *Environment*, *Social* y *Governance*, por lo que los indicadores del archivo **Other** conservan valores faltantes de manera intencional. En consecuencia, estos valores no representan errores en el proceso de integración de la base de datos.

In [34]:
# VALIDACIÓN FINAL DE LOS DICCIONARIOS

# Verificamos que todas las variables de la base estén documentadas
print("Variables en la base:")

print(
    len(base_final.columns)
)

print("\nVariables en el diccionario:")

print(
    len(diccionario_variables)
)


# Comparamos ambos conjuntos de variables
print("\n¿Las variables coinciden?")

print(
    set(base_final.columns)
    ==
    set(diccionario_variables["Variable"])
)


# Verificamos que todos los indicadores estén documentados
print("\nIndicadores en la base:")

print(
    base_final["Metric"]
    .nunique()
)

print("\nIndicadores en el diccionario:")

print(
    len(diccionario_indicadores)
)


# Comparamos ambos conjuntos de indicadores
print("\n¿Los indicadores coinciden?")

print(
    set(
        base_final["Metric"]
        .str.replace(
            "Fashion Revolution+",
            "",
            regex=False
        )
    )
    ==
    set(diccionario_indicadores["Indicador"])
)


# Verificamos que no existan significados faltantes
print("\nSignificados faltantes:")

print(
    diccionario_indicadores["Significado"]
    .isna()
    .sum()
)


# Verificamos que no existan categorías temáticas faltantes
print("\nCategorías temáticas faltantes:")

print(
    diccionario_indicadores["Categoría temática"]
    .isna()
    .sum()
)


# Verificamos que no existan tipos de respuesta faltantes
print("\nTipos de respuesta faltantes:")

print(
    diccionario_indicadores["Tipo de respuesta"]
    .isna()
    .sum()
)


# Verificamos que no existan indicadores duplicados
print("\nIndicadores duplicados:")

print(
    diccionario_indicadores["Indicador"]
    .duplicated()
    .sum()
)


# Mostramos la distribución de categorías temáticas
print("\nCategorías temáticas:")

print(
    diccionario_indicadores["Categoría temática"]
    .value_counts()
)


# Mostramos la distribución de tipos de respuesta
print("\nTipos de respuesta:")

print(
    diccionario_indicadores["Tipo de respuesta"]
    .value_counts()
)

Variables en la base:
12

Variables en el diccionario:
12

¿Las variables coinciden?
True

Indicadores en la base:
130

Indicadores en el diccionario:
130

¿Los indicadores coinciden?
True

Significados faltantes:
0

Categorías temáticas faltantes:
0

Tipos de respuesta faltantes:
0

Indicadores duplicados:
0

Categorías temáticas:
Categoría temática
Social          53
Environment     47
Governance      26
Traceability     4
Name: count, dtype: int64

Tipos de respuesta:
Tipo de respuesta
Sí / No               110
Lista de elementos     10
Puntaje                 6
Porcentaje              3
Texto                   1
Name: count, dtype: int64


### Exportación de la base de datos y los diccionarios

In [35]:
# Exportamos la base de datos integrada

base_final.to_csv(
    "../processed/base_integrada.csv",
    index=False
)


# Exportamos el diccionario de variables

diccionario_variables.to_excel(
    "../processed/diccionario_variables.xlsx",
    index=False
)


# Exportamos el diccionario de indicadores

diccionario_indicadores.to_excel(
    "../processed/diccionario_indicadores.xlsx",
    index=False
)


# Confirmamos que la exportación finalizó correctamente

print("Archivos exportados correctamente.")

Archivos exportados correctamente.
